# 1 DATA PREPROCESSING

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/AI_DA_PREDICTIVE MAINTAINENCE/Dataset.zip" -d "/content/Dataset/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/MyDrive/AI_DA_PREDICTIVE MAINTAINENCE/Dataset.zip
replace /content/Dataset/Dataset/train_operational_readouts.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/Dataset/Dataset/train_specifications.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/Dataset/Dataset/train_tte.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/Dataset/Dataset/validation_labels.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/Dataset/Dataset/validation_operational_readouts.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no
replace /content/Dataset/Dataset/validation_specifications.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: no


In [ ]:
import pandas as pd

df_train_ops = pd.read_csv('/content/Dataset/Dataset/train_operational_readouts.csv')
df_train_specs = pd.read_csv('/content/Dataset/Dataset/train_specifications.csv')
df_train_tte = pd.read_csv('/content/Dataset/Dataset/train_tte.csv')

df_val_ops = pd.read_csv('/content/Dataset/Dataset/validation_operational_readouts.csv')
df_val_specs = pd.read_csv('/content/Dataset/Dataset/validation_specifications.csv')
df_val_labels = pd.read_csv('/content/Dataset/Dataset/validation_labels.csv')

# Merge 1: Attach specifications to the operational readouts
train_merged_1 = pd.merge(df_train_ops, df_train_specs, on='vehicle_id', how='left')

# Merge 2: Attach the target labels (time to event / repair status)
master_train = pd.merge(train_merged_1, df_train_tte, on='vehicle_id', how='left')

print(f"Master Train Shape: {master_train.shape}")


# ==========================================
# 2. MERGING THE VALIDATION DATA
# ==========================================

# Merge 1: Attach specifications to the operational readouts
val_merged_1 = pd.merge(df_val_ops, df_val_specs, on='vehicle_id', how='left')

# Merge 2: Attach the target labels (class label)
master_validation = pd.merge(val_merged_1, df_val_labels, on='vehicle_id', how='left')

print(f"Master Validation Shape: {master_validation.shape}")

Master Train Shape: (1122452, 117)
Master Validation Shape: (196227, 116)


## 1.1 MERGING THE TRAINING DATA

In [ ]:
list(master_train.columns)
master_train.head()
list(master_validation.columns)
master_validation.head()

,vehicle_id,time_step,171_0,666_0,427_0,837_0,167_0,167_1,167_2,167_3,...,397_35,Spec_0,Spec_1,Spec_2,Spec_3,Spec_4,Spec_5,Spec_6,Spec_7,class_label
0,10,3.0,46590.0,3696.0,2038959.0,1450.0,0.0,273826.0,339584.0,99834.0,...,0.0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat1,Cat0,Cat1,0
1,10,7.4,127110.0,16716.0,6501456.0,4660.0,0.0,635642.0,609742.0,288036.0,...,1005.0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat1,Cat0,Cat1,0
2,10,8.0,144015.0,19596.0,7327918.0,5230.0,0.0,654780.0,665756.0,300174.0,...,1500.0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat1,Cat0,Cat1,0
3,10,12.0,187560.0,24264.0,9286082.0,7420.0,2647.0,831628.0,794332.0,360066.0,...,1591.0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat1,Cat0,Cat1,0
4,10,12.2,187575.0,24264.0,9286082.0,7420.0,NaN,NaN,NaN,NaN,...,1592.0,Cat0,Cat0,Cat0,Cat0,Cat0,Cat1,Cat0,Cat1,0


## 1.2 Missing Value Analysis

In [ ]:
#2 Missing Value Analysis
master_train = master_train.ffill()
master_train = master_train.bfill()
master_validation = master_validation.ffill()
master_validation = master_validation.bfill()
print("Missing values in Training Data:", master_train.isna().sum().sum())
print("Missing values in Validation Data:", master_validation.isna().sum().sum())

Missing values in Training Data: 0
Missing values in Validation Data: 0


## 1.3 Feature Scaling (Normalization)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# 1. Identify and Encode the Categorical 'Spec' columns FIRST
spec_cols = [col for col in master_train.columns if col.startswith('Spec_')]

oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
master_train[spec_cols] = oe.fit_transform(master_train[spec_cols].astype(str))
master_validation[spec_cols] = oe.transform(master_validation[spec_cols].astype(str))

# 2. Identify the true sensor columns for scaling
common_cols = [col for col in master_train.columns if col in master_validation.columns]
numeric_cols = master_train[common_cols].select_dtypes(include=['float64', 'int64']).columns

# EXPLICITLY exclude the Spec columns we just encoded, plus IDs and targets
cols_to_exclude = ['vehicle_id', 'time_step', 'class_label', 'length_of_study_time_step', 'in_study_repair'] + spec_cols
features_to_scale = [col for col in numeric_cols if col not in cols_to_exclude]

# 3. Apply Z-Score Standardization strictly to the sensors
scaler = StandardScaler()
master_train[features_to_scale] = scaler.fit_transform(master_train[features_to_scale])
master_validation[features_to_scale] = scaler.transform(master_validation[features_to_scale])

print("Preprocessing Complete!")
print("Mean of scaled sensor 171_0 (should be ~0):", np.round(master_train['171_0'].mean(), 5))
print("Spec_0 unique encoded classes in Train:", master_train['Spec_0'].unique()[:5])

Preprocessing Complete!
Mean of scaled sensor 171_0 (should be ~0): 0.0
Spec_0 unique encoded classes in Train: [0. 1. 2.]


In [ ]:
# Replace your current threshold-based labeling with this
vehicles_that_fail = master_train[master_train['in_study_repair'] == 1]['vehicle_id'].unique()

master_train['class_label'] = master_train['vehicle_id'].isin(vehicles_that_fail).astype(int)

print("Training label distribution after fix:")
print(master_train['class_label'].value_counts())
print(f"Train failure rate: {master_train['class_label'].mean():.4f}")

Training label distribution after fix:
class_label
0    1016600
1     105852
Name: count, dtype: int64
Train failure rate: 0.0943


/tmp/ipython-input-2978060219.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_train['class_label'] = master_train['vehicle_id'].isin(vehicles_that_fail).astype(int)


In [ ]:
print(np.round(master_train[features_to_scale].mean()[:10], 5))

171_0    0.0
666_0   -0.0
427_0    0.0
837_0    0.0
167_0    0.0
167_1   -0.0
167_2    0.0
167_3   -0.0
167_4   -0.0
167_5   -0.0
dtype: float64


# 2 FEATURE ENGINEERING AND EXTRACTION

## Target Variable Formulation

In [ ]:
master_train['RUL'] = master_train['length_of_study_time_step'] - master_train['time_step']

vehicles_that_fail = master_train[master_train['in_study_repair'] == 1]['vehicle_id'].unique()
master_train['class_label'] = master_train['vehicle_id'].isin(vehicles_that_fail).astype(int)

print("Training label distribution after fix:")
print(master_train['class_label'].value_counts())
print(f"Train failure rate: {master_train['class_label'].mean():.4f}")

Training label distribution after fix:
class_label
0    1016600
1     105852
Name: count, dtype: int64
Train failure rate: 0.0943


/tmp/ipython-input-2379747802.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_train['RUL'] = master_train['length_of_study_time_step'] - master_train['time_step']


## Algorithm 1: Statistical Time-Domain Extraction

In [ ]:
import pandas as pd
import numpy as np

# Rolling mean and std for key sensors — applied to BOTH train and validation
feature_cols = ['171_0', '666_0', '427_0']

for col in feature_cols:
    master_train[f'{col}_rolling_mean'] = master_train.groupby('vehicle_id')[col].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean())
    master_train[f'{col}_rolling_std'] = master_train.groupby('vehicle_id')[col].transform(
        lambda x: x.rolling(window=5, min_periods=1).std().fillna(0))

    master_validation[f'{col}_rolling_mean'] = master_validation.groupby('vehicle_id')[col].transform(
        lambda x: x.rolling(window=5, min_periods=1).mean())
    master_validation[f'{col}_rolling_std'] = master_validation.groupby('vehicle_id')[col].transform(
        lambda x: x.rolling(window=5, min_periods=1).std().fillna(0))

print('Statistical rolling features extracted for Train and Validation.')
print(master_train[['171_0_rolling_mean', '171_0_rolling_std']].head())


/tmp/ipython-input-4109557101.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_train[f'{col}_rolling_mean'] = master_train.groupby('vehicle_id')[col].transform(
/tmp/ipython-input-4109557101.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_train[f'{col}_rolling_std'] = master_train.groupby('vehicle_id')[col].transform(
/tmp/ipython-input-4109557101.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider

Statistical rolling features extracted for Train and Validation.
   171_0_rolling_mean  171_0_rolling_std
0           -1.179871           0.000000
1           -1.179871           0.000000
2           -1.160154           0.034151
3           -1.148187           0.036748
4           -1.140242           0.036447


/tmp/ipython-input-4109557101.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  master_validation[f'{col}_rolling_std'] = master_validation.groupby('vehicle_id')[col].transform(


## Algorithm 2: Mutual Information Feature Selection

In [ ]:
from sklearn.feature_selection import mutual_info_classif

print('Calculating Mutual Information scores...')

# MI computed on training data only, against binary class_label target
mi_scores = mutual_info_classif(master_train[features_to_scale], master_train['class_label'], random_state=42)
mi_series = pd.Series(mi_scores, index=features_to_scale)
top_20_features = mi_series.nlargest(20).index.tolist()

print('\nTop 20 Features selected by Mutual Information:')
print(top_20_features)


Calculating Mutual Information scores...


In [ ]:
# Verify master_validation has multiple rows per vehicle
print("Rows per vehicle in validation (sample):")
print(master_validation.groupby('vehicle_id').size().describe())

# Check labels BEFORE expansion
print("\nBefore expansion:", master_validation['class_label'].value_counts().to_dict())

# Apply expansion
master_validation['class_label'] = master_validation.groupby('vehicle_id')['class_label'].transform(
    lambda x: x.iloc[::-1].rolling(window=30, min_periods=1).max().iloc[::-1]
).astype(int)

# Check labels AFTER expansion
print("After expansion:", master_validation['class_label'].value_counts().to_dict())

Rows per vehicle in validation (sample):
count    5046.000000
mean       38.887634
std        25.050559
min         1.000000
25%        20.000000
50%        34.000000
75%        54.000000
max       209.000000
dtype: float64

Before expansion: {0: 191150, 4: 2997, 3: 948, 2: 600, 1: 532}
After expansion: {0: 191150, 4: 2997, 3: 948, 2: 600, 1: 532}


# aLGORITHM 3 Sequential Windowing: 2D to 3D Tensor Conversion

In [ ]:
import numpy as np

master_validation['class_label'] = (master_validation['class_label'] > 0).astype(int)

def create_sequences(data, seq_length, feature_cols, target_col):
    sequences, labels = [], []
    for vehicle_id, group in data.groupby('vehicle_id'):
        group_features = group[feature_cols].values
        group_labels = group[target_col].values
        if len(group) > seq_length:
            for i in range(len(group) - seq_length):
                sequences.append(group_features[i:i+seq_length])
                labels.append(group_labels[i+seq_length])
    return np.array(sequences), np.array(labels)

seq_length = 10

X_train_final, y_train_final = create_sequences(master_train, seq_length, top_20_features, 'class_label')
X_val_final, y_val_final = create_sequences(master_validation, seq_length, top_20_features, 'class_label')

print(f'X_train: {X_train_final.shape}, y_train: {y_train_final.shape}')
print(f'X_val:   {X_val_final.shape},   y_val:   {y_val_final.shape}')
print('Unique train labels:', np.unique(y_train_final))
print('Unique val labels:  ', np.unique(y_val_final))

X_train: (889782, 10, 20), y_train: (889782,)
X_val:   (147307, 10, 20),   y_val:   (147307,)
Unique train labels: [0 1]
Unique val labels:   [0 1]


Build and Compile the LSTM Architecture

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam

model = Sequential()

model.add(LSTM(64, return_sequences=True, input_shape=(X_train_final.shape[1], X_train_final.shape[2])))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(LSTM(32, return_sequences=False))
model.add(BatchNormalization())
model.add(Dropout(0.2))

model.add(Dense(16, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

custom_adam = Adam(learning_rate=0.0005)

model.compile(
    optimizer=custom_adam,
    loss='binary_crossentropy',
    metrics=['accuracy', 'precision', 'recall']
)

In [ ]:
print(f"Train failure rate: {y_train_final.mean():.4f}")
print(f"Val failure rate:   {y_val_final.mean():.4f}")

Train failure rate: 0.0936
Val failure rate:   0.0253


Train the Model

In [ ]:
import numpy as np
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import EarlyStopping

# 1. Calculate the Class Weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_final)
class_weights_dict = dict(zip(classes, weights))
print(class_weights_dict)

# 2. Setup Early Stopping
early_stop = EarlyStopping(
    monitor='val_recall',
    patience=10,
    restore_best_weights=True,
    mode='max',  # higher recall is better
    verbose=1
)

# 3. Train with Class Weights!
history = model.fit(
    X_train_final, y_train_final,
    epochs=50,
    batch_size=64,
    validation_data=(X_val_final, y_val_final),
    callbacks=[early_stop],
    class_weight=class_weights_dict,
    verbose=1
)

Applying Class Weights to force the AI to pay attention to failures:
Weight for Class 0 (Healthy): 1.0000
Weight for Class 1 (Failure): 4.0000

Epoch 1/50
13903/13903 ━━━━━━━━━━━━━━━━━━━━ 160s 11ms/step - accuracy: 0.8852 - loss: 0.7423 - precision: 0.1987 - recall: 0.0726 - val_accuracy: 0.8944 - val_loss: 0.3715 - val_precision: 0.0263 - val_recall: 0.0883
Epoch 2/50
13903/13903 ━━━━━━━━━━━━━━━━━━━━ 154s 11ms/step - accuracy: 0.8693 - loss: 0.6822 - precision: 0.2762 - recall: 0.2464 - val_accuracy: 0.9229 - val_loss: 0.3002 - val_precision: 0.0211 - val_recall: 0.0451
Epoch 3/50
13903/13903 ━━━━━━━━━━━━━━━━━━━━ 153s 11ms/step - accuracy: 0.8696 - loss: 0.6381 - precision: 0.3172 - recall: 0.3395 - val_accuracy: 0.8467 - val_loss: 0.3743 - val_precision: 0.0354 - val_recall: 0.1929
Epoch 4/50
13903/13903 ━━━━━━━━━━━━━━━━━━━━ 155s 11ms/step - accuracy: 0.8759 - loss: 0.5992 - precision: 0.3511 - recall: 0.3931 - val_accuracy: 0.8709 - val_loss: 0.3529 - val_precision: 0.0271 - val_rec

KeyboardInterrupt: 

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Use 0.25 threshold instead of default 0.5
y_pred = (model.predict(X_val_final) > 0.25).astype(int)

print("Classification Report:")
print(classification_report(y_val_final, y_pred, target_names=['Healthy', 'Failure']))

print("Confusion Matrix:")
print(confusion_matrix(y_val_final, y_pred))

4604/4604 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step
Classification Report:
              precision    recall  f1-score   support

     Healthy       0.97      0.84      0.90    143580
     Failure       0.02      0.15      0.04      3727

    accuracy                           0.82    147307
   macro avg       0.50      0.50      0.47    147307
weighted avg       0.95      0.82      0.88    147307

Confusion Matrix:
[[120454  23126]
 [  3159    568]]


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# Plot 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('LSTM Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)

# Plot 2: Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('LSTM Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import shap
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.experimental.numpy.experimental_enable_numpy_behavior()

background_indices = np.random.choice(X_train_final.shape[0], 100, replace=False)
background = tf.cast(X_train_final[background_indices], tf.float32)

explainer = shap.GradientExplainer(model, background)
test_sequences = tf.cast(X_val_final[:50], tf.float32)

shap_values = explainer.shap_values(test_sequences)

if isinstance(shap_values, list):
    shap_values = shap_values[0]

if len(shap_values.shape) == 4:
    shap_values = shap_values[:, :, :, 0]

shap_values_2d = np.abs(shap_values).mean(axis=1)
global_shap = np.mean(shap_values_2d, axis=0)

sorted_idx = np.argsort(global_shap)
sorted_features = [top_20_features[i] for i in sorted_idx]
sorted_values = global_shap[sorted_idx]

plt.figure(figsize=(10, 8))
plt.barh(sorted_features, sorted_values, color='#007acc')
plt.xlabel('Mean |SHAP Value| (Average Impact on Prediction)')
plt.title('Global Feature Importance for Scania LSTM')
plt.tight_layout()
plt.show()